In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load your cleaned dataset (update the path if needed)
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# Rename columns for compatibility
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989"
})

# Convert necessary columns to numeric
for col in [
    'church_persistence_index', 'income_1989_real_2023', 'pct_hs_1990',
    'pop_2023', 'firms_2022', 'firms_1989'
]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop missing or invalid values
df = df.dropna(subset=[
    'income_1989_real_2023', 'pct_hs_1990', 'pop_2023',
    'firms_2022', 'firms_1989', 'State'
])
df = df[(df['firms_2022'] > 0) & (df['pop_2023'] > 0) & (df['firms_1989'] > 0)]

# Create log-transformed and scaled variables
df['log_pop_2023'] = np.log(df['pop_2023'])
df['log_firms_1989'] = np.log(df['firms_1989'])

scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run the placebo regression: log(pop_2023) ~ persistence + controls
placebo_model = smf.ols(
    formula="""
        log_pop_2023 ~ 
        church_persistence_index + 
        income_1989_scaled + 
        pct_hs_1990_scaled + 
        log_firms_1989 + 
        C(State)
    """,
    data=df
).fit(cov_type='HC3')

# Print regression summary
print(placebo_model.summary())


                            OLS Regression Results                            
Dep. Variable:           log_pop_2023   R-squared:                       0.949
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     935.3
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        11:30:41   Log-Likelihood:                -850.10
No. Observations:                2968   AIC:                             1804.
Df Residuals:                    2916   BIC:                             2116.
Df Model:                          51                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               